# X3: Debugging and Monitoring RL Systems

**Learning Objectives:**
- Diagnose common RL failure modes
- Implement comprehensive logging and visualization
- Debug policy and value function issues
- Detect and fix training instabilities
- Monitor distribution shift
- Build debugging toolkit

**Prerequisites:** All main curriculum, X1-X2

**Key Insight**: "RL is notoriously hard to debug. What works for supervised learning often doesn't apply." - John Schulman

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import deque, defaultdict
import warnings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Common RL Failure Modes

Understanding what can go wrong.

In [ ]:
class RLFailureModes:
    """Catalog of common RL failures and diagnostics."""
    
    FAILURE_MODES = {
        'no_learning': {
            'symptoms': [
                'Reward stays flat from beginning',
                'Policy entropy stays constant or increases',
                'Value function predictions near zero'
            ],
            'causes': [
                'Learning rate too low',
                'Wrong reward scaling',
                'Incorrect observation normalization',
                'Bug in reward function'
            ],
            'fixes': [
                'Increase learning rate (try 10x)',
                'Check reward is non-zero',
                'Print observations and verify normalization',
                'Unit test reward function'
            ]
        },
        
        'policy_collapse': {
            'symptoms': [
                'Policy entropy drops to near zero',
                'Agent takes same action repeatedly',
                'Early reward spike then plateau',
                'Gradient norms become very large then zero'
            ],
            'causes': [
                'Learning rate too high',
                'Insufficient exploration (entropy coef too low)',
                'Reward hacking / degenerate solution',
                'Trust region violations (PPO clip too large)'
            ],
            'fixes': [
                'Decrease learning rate',
                'Increase entropy coefficient',
                'Add exploration bonuses',
                'Reduce PPO clip epsilon',
                'Check if reward function has unintended solutions'
            ]
        },
        
        'value_divergence': {
            'symptoms': [
                'Value loss explodes',
                'Value predictions become extremely large',
                'Training becomes unstable / crashes',
                'NaN or Inf in gradients'
            ],
            'causes': [
                'Value function learning rate too high',
                'Rewards not normalized',
                'Bootstrapping errors compound',
                'Missing gradient clipping'
            ],
            'fixes': [
                'Normalize rewards (running mean/std)',
                'Clip value loss',
                'Add gradient clipping',
                'Reduce value function learning rate',
                'Use value function clipping (PPO)'
            ]
        },
        
        'distribution_shift': {
            'symptoms': [
                'Performance degrades after initial improvement',
                'High variance in episode returns',
                'Policy forgets earlier behaviors',
                'Periodic performance crashes'
            ],
            'causes': [
                'Non-stationary environment',
                'Catastrophic forgetting',
                'Data distribution changes during training',
                'Replay buffer staleness'
            ],
            'fixes': [
                'Use larger replay buffer',
                'Regularize policy updates (KL penalty)',
                'Sample from full buffer history',
                'Add auxiliary tasks for regularization'
            ]
        },
        
        'slow_convergence': {
            'symptoms': [
                'Learning but very slowly',
                'Needs 10x more samples than expected',
                'Gradual improvement but inefficient'
            ],
            'causes': [
                'Network too small',
                'Learning rate too conservative',
                'Poor exploration',
                'Inefficient batch size'
            ],
            'fixes': [
                'Increase network capacity',
                'Tune learning rate upward',
                'Add curiosity / intrinsic rewards',
                'Increase batch size',
                'Use better optimizer (Adam > SGD)'
            ]
        }
    }
    
    @classmethod
    def diagnose(cls, metrics):
        """Diagnose failure mode from metrics.
        
        Args:
            metrics: Dict with 'reward_trend', 'entropy_trend', 'value_loss', etc.
        
        Returns:
            List of likely failure modes
        """
        diagnoses = []
        
        # No learning
        if metrics.get('reward_std', 1.0) < 0.1:
            diagnoses.append('no_learning')
        
        # Policy collapse
        if metrics.get('entropy', 1.0) < 0.01:
            diagnoses.append('policy_collapse')
        
        # Value divergence
        if metrics.get('value_loss', 0) > 1000:
            diagnoses.append('value_divergence')
        
        # Distribution shift
        if metrics.get('reward_variance', 0) > metrics.get('reward_mean', 1) ** 2:
            diagnoses.append('distribution_shift')
        
        return diagnoses
    
    @classmethod
    def display_guide(cls):
        """Display debugging guide."""
        print("RL DEBUGGING GUIDE")
        print("=" * 80)
        
        for mode_name, mode_info in cls.FAILURE_MODES.items():
            print(f"\n{'='*80}")
            print(f"FAILURE MODE: {mode_name.upper().replace('_', ' ')}")
            print(f"{'='*80}")
            
            print("\nSymptoms:")
            for symptom in mode_info['symptoms']:
                print(f"  • {symptom}")
            
            print("\nCauses:")
            for cause in mode_info['causes']:
                print(f"  • {cause}")
            
            print("\nFixes:")
            for fix in mode_info['fixes']:
                print(f"  ✓ {fix}")

# Display guide
RLFailureModes.display_guide()

print("\n" + "="*80)
print("Failure mode catalog created")

## 2. Comprehensive Logging

Log everything you might need.

In [ ]:
class RLLogger:
    """Comprehensive RL training logger."""
    
    def __init__(self):
        self.metrics = defaultdict(list)
        self.episodes = []
        self.step = 0
    
    def log_episode(self, episode_data):
        """Log complete episode data."""
        self.episodes.append({
            'step': self.step,
            **episode_data
        })
    
    def log_metric(self, name, value, step=None):
        """Log scalar metric."""
        if step is None:
            step = self.step
        self.metrics[name].append((step, value))
    
    def log_training_step(self, policy_loss, value_loss, entropy, 
                         grad_norm=None, kl_div=None):
        """Log training step metrics."""
        self.log_metric('policy_loss', policy_loss)
        self.log_metric('value_loss', value_loss)
        self.log_metric('entropy', entropy)
        
        if grad_norm is not None:
            self.log_metric('grad_norm', grad_norm)
        if kl_div is not None:
            self.log_metric('kl_div', kl_div)
        
        self.step += 1
    
    def get_summary(self, last_n=100):
        """Get summary statistics."""
        summary = {}
        
        # Episode statistics
        if self.episodes:
            recent_episodes = self.episodes[-last_n:]
            rewards = [ep['reward'] for ep in recent_episodes]
            lengths = [ep['length'] for ep in recent_episodes]
            
            summary['episode'] = {
                'reward_mean': np.mean(rewards),
                'reward_std': np.std(rewards),
                'reward_min': np.min(rewards),
                'reward_max': np.max(rewards),
                'length_mean': np.mean(lengths),
                'count': len(recent_episodes)
            }
        
        # Training metrics
        for metric_name in ['policy_loss', 'value_loss', 'entropy', 'grad_norm']:
            if metric_name in self.metrics:
                recent_values = [v for _, v in self.metrics[metric_name][-last_n:]]
                if recent_values:
                    summary[metric_name] = {
                        'mean': np.mean(recent_values),
                        'std': np.std(recent_values),
                        'latest': recent_values[-1]
                    }
        
        return summary
    
    def plot_training(self, figsize=(15, 10)):
        """Plot comprehensive training metrics."""
        fig, axes = plt.subplots(3, 2, figsize=figsize)
        fig.suptitle('RL Training Diagnostics', fontsize=16)
        
        # Episode rewards
        if self.episodes:
            steps = [ep['step'] for ep in self.episodes]
            rewards = [ep['reward'] for ep in self.episodes]
            
            axes[0, 0].plot(steps, rewards, alpha=0.3, label='Raw')
            
            # Moving average
            if len(rewards) >= 10:
                window = min(20, len(rewards) // 5)
                ma_rewards = np.convolve(rewards, np.ones(window)/window, mode='valid')
                ma_steps = steps[window-1:]
                axes[0, 0].plot(ma_steps, ma_rewards, linewidth=2, label=f'MA({window})')
            
            axes[0, 0].set_xlabel('Step')
            axes[0, 0].set_ylabel('Episode Reward')
            axes[0, 0].set_title('Episode Rewards')
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)
        
        # Policy loss
        if 'policy_loss' in self.metrics:
            steps, values = zip(*self.metrics['policy_loss'])
            axes[0, 1].plot(steps, values)
            axes[0, 1].set_xlabel('Step')
            axes[0, 1].set_ylabel('Policy Loss')
            axes[0, 1].set_title('Policy Loss')
            axes[0, 1].grid(True, alpha=0.3)
        
        # Value loss
        if 'value_loss' in self.metrics:
            steps, values = zip(*self.metrics['value_loss'])
            axes[1, 0].plot(steps, values)
            axes[1, 0].set_xlabel('Step')
            axes[1, 0].set_ylabel('Value Loss')
            axes[1, 0].set_title('Value Loss')
            axes[1, 0].set_yscale('log')
            axes[1, 0].grid(True, alpha=0.3)
        
        # Entropy
        if 'entropy' in self.metrics:
            steps, values = zip(*self.metrics['entropy'])
            axes[1, 1].plot(steps, values)
            axes[1, 1].set_xlabel('Step')
            axes[1, 1].set_ylabel('Entropy')
            axes[1, 1].set_title('Policy Entropy (Exploration)')
            axes[1, 1].grid(True, alpha=0.3)
        
        # Gradient norms
        if 'grad_norm' in self.metrics:
            steps, values = zip(*self.metrics['grad_norm'])
            axes[2, 0].plot(steps, values)
            axes[2, 0].set_xlabel('Step')
            axes[2, 0].set_ylabel('Gradient Norm')
            axes[2, 0].set_title('Gradient Norms')
            axes[2, 0].set_yscale('log')
            axes[2, 0].grid(True, alpha=0.3)
        
        # KL divergence (if available)
        if 'kl_div' in self.metrics:
            steps, values = zip(*self.metrics['kl_div'])
            axes[2, 1].plot(steps, values)
            axes[2, 1].set_xlabel('Step')
            axes[2, 1].set_ylabel('KL Divergence')
            axes[2, 1].set_title('KL Divergence (Policy Change)')
            axes[2, 1].grid(True, alpha=0.3)
        else:
            axes[2, 1].text(0.5, 0.5, 'No KL data', 
                          ha='center', va='center', transform=axes[2, 1].transAxes)
        
        plt.tight_layout()
        return fig

# Example usage
logger = RLLogger()

# Simulate training
print("Simulating training with logging...")
for step in range(200):
    # Simulate training step
    policy_loss = np.random.gamma(2, 0.5) + 0.1
    value_loss = np.random.gamma(3, 10) + 1.0
    entropy = max(0.01, 1.5 - step * 0.005 + np.random.normal(0, 0.1))
    grad_norm = np.random.lognormal(0, 0.5)
    
    logger.log_training_step(policy_loss, value_loss, entropy, grad_norm)
    
    # Simulate episodes
    if step % 10 == 0:
        reward = step * 0.5 + np.random.normal(0, 10)
        logger.log_episode({
            'reward': reward,
            'length': np.random.randint(50, 200)
        })

# Get summary
summary = logger.get_summary()
print("\nTraining Summary:")
print(f"  Episodes: {summary['episode']['count']}")
print(f"  Reward: {summary['episode']['reward_mean']:.2f} ± {summary['episode']['reward_std']:.2f}")
print(f"  Entropy: {summary['entropy']['latest']:.3f}")
print(f"  Value loss: {summary['value_loss']['latest']:.2f}")

# Plot
logger.plot_training()
plt.show()

print("\nComprehensive logging implemented")

## 3. Debugging Checklist

Systematic debugging workflow.

In [ ]:
class RLDebuggingChecklist:
    """Interactive debugging checklist for RL."""
    
    CHECKLIST = [
        {
            'category': 'Environment',
            'checks': [
                'Reward function returns non-zero values',
                'Observations are within expected range',
                'Episode termination works correctly',
                'Random policy achieves non-zero reward',
                'Environment is deterministic (if expected)',
                'Action space is correctly defined'
            ]
        },
        {
            'category': 'Data Processing',
            'checks': [
                'Observations are normalized (mean ~0, std ~1)',
                'Rewards are scaled appropriately',
                'No NaN or Inf in observations',
                'Batch shapes are correct',
                'Data types are correct (float32, not float64)'
            ]
        },
        {
            'category': 'Model Architecture',
            'checks': [
                'Network has sufficient capacity',
                'Activation functions are appropriate',
                'Output dimensions match action space',
                'Shared vs separate networks (policy/value)',
                'Initialization is reasonable'
            ]
        },
        {
            'category': 'Optimization',
            'checks': [
                'Learning rate is in reasonable range (1e-5 to 1e-3)',
                'Gradients are flowing (not zero or exploding)',
                'Gradient clipping is enabled',
                'Optimizer is appropriate (Adam usually good)',
                'Batch size is reasonable (32-512)'
            ]
        },
        {
            'category': 'Algorithm',
            'checks': [
                'Advantage estimation is correct',
                'Value function is learning',
                'Policy entropy is non-zero (exploration)',
                'Trust region is respected (PPO clip)',
                'Discount factor (gamma) is appropriate for horizon'
            ]
        },
        {
            'category': 'Training Loop',
            'checks': [
                'Sufficient training steps',
                'Evaluation frequency is reasonable',
                'Metrics are being logged',
                'Checkpointing works',
                'Reproducibility (seeds set)'
            ]
        }
    ]
    
    @classmethod
    def display(cls):
        """Display complete checklist."""
        print("="*80)
        print("RL DEBUGGING CHECKLIST")
        print("="*80)
        print("\nBefore assuming algorithm bugs, verify these systematically:\n")
        
        for item in cls.CHECKLIST:
            print(f"\n{'='*80}")
            print(f"{item['category'].upper()}")
            print(f"{'='*80}")
            
            for i, check in enumerate(item['checks'], 1):
                print(f"  [{' '}] {i}. {check}")
        
        print("\n" + "="*80)
        print("DEBUGGING TIPS:")
        print("="*80)
        print("1. Start simple: Random policy → Simplest baseline → Full algorithm")
        print("2. Overfit single episode before scaling up")
        print("3. Compare with known-good implementation (baselines)")
        print("4. Visualize policy behavior, don't just look at numbers")
        print("5. Test each component in isolation")
    
    @classmethod
    def run_automated_checks(cls, env, policy, config):
        """Run automated sanity checks."""
        print("Running automated sanity checks...\n")
        
        results = {}
        
        # Check 1: Reward function
        print("[1/6] Checking reward function...")
        try:
            state = env.reset()
            action = env.action_space.sample()
            next_state, reward, done, _ = env.step(action)
            
            reward_check = abs(reward) > 0 or done
            results['reward_function'] = 'PASS' if reward_check else 'WARNING'
            print(f"  Reward: {reward}, Done: {done} - {'PASS' if reward_check else 'WARNING'}")
        except Exception as e:
            results['reward_function'] = 'FAIL'
            print(f"  FAIL: {e}")
        
        # Check 2: Observation range
        print("[2/6] Checking observation range...")
        try:
            obs_samples = []
            for _ in range(100):
                state = env.reset()
                obs_samples.append(state)
            
            obs_array = np.array(obs_samples)
            obs_mean = np.mean(obs_array)
            obs_std = np.std(obs_array)
            
            normalized = abs(obs_mean) < 5 and 0.1 < obs_std < 10
            results['observations'] = 'PASS' if normalized else 'WARNING'
            print(f"  Mean: {obs_mean:.2f}, Std: {obs_std:.2f} - {'PASS' if normalized else 'WARNING'}")
        except Exception as e:
            results['observations'] = 'FAIL'
            print(f"  FAIL: {e}")
        
        # Check 3: Policy forward pass
        print("[3/6] Checking policy forward pass...")
        try:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                output = policy(state_tensor)
            
            has_nan = torch.isnan(output).any().item()
            has_inf = torch.isinf(output).any().item()
            
            results['policy'] = 'PASS' if not (has_nan or has_inf) else 'FAIL'
            print(f"  Output shape: {output.shape}, NaN: {has_nan}, Inf: {has_inf} - {'PASS' if not (has_nan or has_inf) else 'FAIL'}")
        except Exception as e:
            results['policy'] = 'FAIL'
            print(f"  FAIL: {e}")
        
        # Check 4: Gradient flow
        print("[4/6] Checking gradient flow...")
        try:
            # Dummy loss
            loss = output.sum()
            loss.backward()
            
            total_norm = 0
            for p in policy.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5
            
            has_grads = total_norm > 0
            results['gradients'] = 'PASS' if has_grads else 'FAIL'
            print(f"  Gradient norm: {total_norm:.4f} - {'PASS' if has_grads else 'FAIL'}")
        except Exception as e:
            results['gradients'] = 'FAIL'
            print(f"  FAIL: {e}")
        
        # Check 5: Learning rate
        print("[5/6] Checking learning rate...")
        lr = config.get('lr', 0)
        reasonable_lr = 1e-5 <= lr <= 1e-2
        results['learning_rate'] = 'PASS' if reasonable_lr else 'WARNING'
        print(f"  LR: {lr} - {'PASS' if reasonable_lr else 'WARNING (outside typical range 1e-5 to 1e-2)'}")
        
        # Check 6: Random policy baseline
        print("[6/6] Testing random policy...")
        try:
            total_reward = 0
            for _ in range(10):
                state = env.reset()
                done = False
                ep_reward = 0
                steps = 0
                while not done and steps < 1000:
                    action = env.action_space.sample()
                    state, reward, done, _ = env.step(action)
                    ep_reward += reward
                    steps += 1
                total_reward += ep_reward
            
            avg_reward = total_reward / 10
            results['random_policy'] = 'PASS'
            print(f"  Average reward: {avg_reward:.2f} - PASS")
        except Exception as e:
            results['random_policy'] = 'FAIL'
            print(f"  FAIL: {e}")
        
        # Summary
        print("\n" + "="*80)
        print("SUMMARY")
        print("="*80)
        
        passes = sum(1 for v in results.values() if v == 'PASS')
        warnings = sum(1 for v in results.values() if v == 'WARNING')
        fails = sum(1 for v in results.values() if v == 'FAIL')
        
        print(f"  PASS: {passes}")
        print(f"  WARNING: {warnings}")
        print(f"  FAIL: {fails}")
        
        if fails == 0:
            print("\n✓ All critical checks passed. Ready to train!")
        else:
            print("\n⚠  Some checks failed. Fix these before training.")
        
        return results

# Display checklist
RLDebuggingChecklist.display()

print("\nDebugging checklist implemented")

## Summary

### Debugging Workflow:

1. **Identify Symptoms**
   - No learning, collapse, divergence, shift, slow convergence
   - Use comprehensive logging

2. **Run Checklist**
   - Environment, data, model, optimization, algorithm, training
   - Automated sanity checks

3. **Test Components**
   - Random policy baseline
   - Overfit single episode
   - Compare with known-good implementation

4. **Visualize**
   - Plot all metrics
   - Watch policy behavior
   - Inspect gradients

5. **Iterate**
   - Fix one issue at a time
   - Re-run checks
   - Document findings

### Key Principles:

✅ **Log everything**: You can't debug what you can't see
✅ **Start simple**: Random → Baseline → Full
✅ **Test in isolation**: Each component separately
✅ **Compare**: Known-good implementations
✅ **Visualize**: Don't just look at numbers

### Common Mistakes:

❌ Not checking reward function first
❌ Assuming algorithm is correct
❌ Not normalizing observations
❌ Ignoring entropy collapse
❌ No gradient clipping

### Next:

**X4**: Real-world applications and complete case studies